----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - IPA - Swap stream - Used as a cache
This notebook demonstrates how to open a FX Cross pricing analytics stream and use it as a real-time data cache.

#### Learn more

To learn more about the LSEG Data Library for Python please join the LSEG Developer Community. By [registering](https://developers.refinitiv.com/iam/register) and [logging](https://developers.refinitiv.com/content/devportal/en_us/initCookie.html) into the LSEG Developer Community portal you will have free access to a number of learning materials 
 on the [LSEG Developer Portal](https://developers.lseg.com/en)

#### Getting Help and Support

If you have any questions regarding using the API, please post them on 
the [LSEG Data Q&A Forum](https://community.developers.refinitiv.com/spaces/321/index.html). 
The LSEG Developer Community will be happy to help. 

## Set the configuration file location
For a better ease of use, you have the option to set initialization parameters of the LSEG Data Library in the _lseg-data.config.json_ configuration file. This file must be located beside your notebook, in your user folder or in a folder defined by the _LD_LIB_CONFIG_PATH_ environment variable. The _LD_LIB_CONFIG_PATH_ environment variable is the option used by this series of examples. The following code sets this environment variable.      

In [1]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [1]:
import lseg.data as ld
from lseg.data.content.ipa.financial_contracts import cross
import datetime

## Open the data session

The open_session() function creates and open sessions based on the information contained in the lseg-data.config.json configuration file. Please edit this file to set the session type and other parameters required for the session you want to open.

In [2]:
ld.open_session()

<lseg.data.session.Definition object at 0x11398ac70 {name='workspace'}>

## Retrieve data

### Define the FxCross

In [3]:
fxcross_def = cross.Definition(
    instrument_tag="USDAUD",
    fx_cross_type=cross.FxCrossType.FX_SPOT,
    fx_cross_code="USDAUD",
    fields=["InstrumentTag", "FxSpot_BidMidAsk", "ErrorCode", "Ccy1SpotDate", "Ccy2SpotDate"],
    extended_params = {
            "marketData": {
                "fxSpots": [
                    {
                        "spotDefinition": {
                            "fxCrossCode": "AUDUSD", 
                            "Source": "Composite"
                        }
                    }
                ]
            }
        }
)

### Get the stream and open it

In [4]:
stream = fxcross_def.get_stream()
stream.open()

<OpenState.Opened: 'Opened'>

### Extract snapshot data from the streaming cache
Once the stream is opened, you can use the get_snapshot method to pull out data from its internal cache. get_snapshot can be called any number of times. As these calls return the latest received values, successive calls to get_snapshot may return different values. Returned DataFrames do not change in real-time, get_snapshot must be called every time your application needs fresh values. 

In [5]:
stream.get_snapshot()

,InstrumentTag,FxSpot_BidMidAsk,ErrorCode,Ccy1SpotDate,Ccy2SpotDate
0,USDAUD,"{'bid': 1.5028554253080855, 'ask': 1.503307276...",,2024-06-17,2024-06-17


### Close the stream

In [6]:
stream.close()

<OpenState.Closed: 'Closed'>

## Close the session

In [7]:
ld.close_session()